# Sesión 1 · Qué es la IA, ML y DL: panorama y usos en ciberseguridad
**IA aplicada a la ciberseguridad · Máster en Ciberseguridad**

Laboratorio: tu primer detector, a mano y aprendido, sobre el dataset **NSL-KDD**.

| Parte | Qué haces |
|---|---|
| A | Preparar el entorno, crear `utils.py` y cargar NSL-KDD |
| B | Mirar los datos antes de modelar |
| C | Regla de analista frente a un árbol de profundidad 1 |
| D | Entrenar con un fichero y probar con otro: sobreajuste |
| E | La trampa de la tasa base |
| F | Ficha de caso de uso (por parejas) |
| G | Ampliación autónoma: aprender un umbral sin scikit-learn |

> **Cómo usar este cuaderno**
> - **Google Colab:** *Archivo › Subir cuaderno*, ejecuta la celda de preparación y sigue en orden. Los ficheros de la sesión (data/, utils.py) se borran al desconectar: vuelve a ejecutar la preparación al retomar.
> - **VS Code o GitHub Codespaces:** abre la carpeta del proyecto, elige el kernel de Python (*Select Kernel*) y ejecuta la celda de preparación.
> - Ejecuta las celdas **en orden** con `Mayús + Intro`. Antes de entregar: reinicia el kernel y ejecuta todo.

## Preparación del entorno

In [ ]:
# Preparación del entorno: ejecuta esta celda la primera (Colab, VS Code o Codespaces)
import sys, os, subprocess, importlib.util, urllib.request

EN_COLAB = "google.colab" in sys.modules
print("Entorno:", "Google Colab" if EN_COLAB else "Jupyter en VS Code / Codespaces")
print("Python:", sys.version.split()[0])

# 1) Librerías: solo se instalan las que falten
paquetes = {"numpy": "numpy", "pandas": "pandas", "sklearn": "scikit-learn",
            "matplotlib": "matplotlib", "seaborn": "seaborn",
            "imblearn": "imbalanced-learn", "joblib": "joblib"}
faltan = [pip for mod, pip in paquetes.items() if importlib.util.find_spec(mod) is None]
if faltan:
    print("Instalando:", ", ".join(faltan))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *faltan])

# 2) Datos: NSL-KDD en la carpeta data/ (se descarga una sola vez)
BASE_URL = "https://raw.githubusercontent.com/defcom17/NSL_KDD/master/"
os.makedirs("data", exist_ok=True)
for nombre in ["KDDTrain+.txt", "KDDTest+.txt"]:
    destino = os.path.join("data", nombre)
    if not os.path.exists(destino):
        urllib.request.urlretrieve(BASE_URL + nombre.replace("+", "%2B"), destino)
    print(f"{nombre}: {os.path.getsize(destino) / 1e6:.1f} MB")

import numpy, pandas, sklearn
print("numpy", numpy.__version__, "| pandas", pandas.__version__,
      "| scikit-learn", sklearn.__version__)

## Parte A · Crea `utils.py` y carga NSL-KDD
La celda siguiente escribe el fichero `utils.py` junto al cuaderno. Lo reutilizaremos en todo el track y lo ampliaremos en la sesión 2.

In [ ]:
%%writefile utils.py
import os, urllib.request
import pandas as pd

NSL_URL = "https://raw.githubusercontent.com/defcom17/NSL_KDD/master/"

NSL_COLS = ["duration","protocol_type","service","flag","src_bytes","dst_bytes","land",
  "wrong_fragment","urgent","hot","num_failed_logins","logged_in","num_compromised",
  "root_shell","su_attempted","num_root","num_file_creations","num_shells",
  "num_access_files","num_outbound_cmds","is_host_login","is_guest_login","count",
  "srv_count","serror_rate","srv_serror_rate","rerror_rate","srv_rerror_rate",
  "same_srv_rate","diff_srv_rate","srv_diff_host_rate","dst_host_count",
  "dst_host_srv_count","dst_host_same_srv_rate","dst_host_diff_srv_rate",
  "dst_host_same_src_port_rate","dst_host_srv_diff_host_rate","dst_host_serror_rate",
  "dst_host_srv_serror_rate","dst_host_rerror_rate","dst_host_srv_rerror_rate",
  "label","difficulty"]


def load_nsl_kdd(split="train", base="data"):
    """Carga NSL-KDD y añade la columna is_attack."""
    name = ("KDDTrain+.txt" if split == "train"
            else "KDDTest+.txt")
    path = os.path.join(base, name)
    if not os.path.exists(path):   # descarga si falta
        os.makedirs(base, exist_ok=True)
        url = NSL_URL + name.replace("+", "%2B")
        urllib.request.urlretrieve(url, path)
    df = pd.read_csv(path, header=None, names=NSL_COLS)
    df["is_attack"] = (df["label"] != "normal").astype(int)
    return df

In [ ]:
from utils import load_nsl_kdd
df = load_nsl_kdd("train")
print(df.shape)
df.head()

**Resultado esperado:** 125.973 filas y 44 columnas (41 características, `label`, `difficulty` e `is_attack`).

## Parte B · Mira los datos antes de modelar

In [ ]:
print(df["is_attack"].value_counts(normalize=True))
print(df["label"].value_counts().head(10))
print(df["protocol_type"].value_counts())
cols = ["duration", "src_bytes", "dst_bytes", "count", "serror_rate"]
df[cols].describe().T

**Responde en esta celda (doble clic para editar):**
1. ¿Qué proporción del dataset es ataque? ¿Sirve como tasa base de una red real?
   *Tu respuesta:*
2. ¿Qué tipos de ataque hay y cuál domina?
   *Tu respuesta:*
3. ¿Qué características tienen escalas muy distintas?
   *Tu respuesta:*

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
df.groupby("is_attack")["serror_rate"].plot(kind="hist", bins=20, alpha=0.6, ax=ax[0], legend=True)
ax[0].set_title("serror_rate: normal (0) vs ataque (1)")
df.groupby("is_attack")["same_srv_rate"].plot(kind="hist", bins=20, alpha=0.6, ax=ax[1], legend=True)
ax[1].set_title("same_srv_rate")
plt.tight_layout(); plt.show()

`serror_rate` (proporción de conexiones con error SYN) separa muy bien los SYN floods (*neptune*). Una característica con poder discriminante es lo que un modelo busca de forma automática.

## Parte C · Regla de analista frente a umbral aprendido

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix

# 1) regla manual
pred_regla = (df["serror_rate"] > 0.5).astype(int)
print("accuracy regla:", accuracy_score(df["is_attack"], pred_regla))
print(confusion_matrix(df["is_attack"], pred_regla))

In [ ]:
# 2) árbol de profundidad 1 (stump)
from sklearn.tree import DecisionTreeClassifier, export_text
X = df[["serror_rate", "same_srv_rate", "count"]]
y = df["is_attack"]
stump = DecisionTreeClassifier(max_depth=1, random_state=0).fit(X, y)
print(export_text(stump, feature_names=list(X.columns)))
print("accuracy stump:", stump.score(X, y))

In [ ]:
# Ejercicio: sube max_depth y anota la exactitud
for depth in [2, 4, 8]:
    m = DecisionTreeClassifier(max_depth=depth, random_state=0).fit(X, y)
    print(depth, round(m.score(X, y), 4))

**Reflexiona:** ¿qué detecta el árbol que tu regla no ve? ¿Qué pasa con la explicabilidad al crecer el árbol?
*Tu respuesta:*

## Parte D · Entrena con un fichero, prueba con otro

In [ ]:
import pandas as pd
tr = load_nsl_kdd("train"); te = load_nsl_kdd("test")
feats = ["duration", "src_bytes", "dst_bytes", "count", "srv_count",
         "serror_rate", "rerror_rate", "same_srv_rate", "diff_srv_rate",
         "dst_host_count", "dst_host_srv_count",
         "dst_host_same_srv_rate", "logged_in"]
rows = []
for depth in [1, 2, 3, 5, 8, 12, 20, None]:
    m = DecisionTreeClassifier(max_depth=depth, random_state=0)
    m.fit(tr[feats], tr["is_attack"])
    rows.append({"max_depth": str(depth),
                 "acc_train": m.score(tr[feats], tr["is_attack"]),
                 "acc_test": m.score(te[feats], te["is_attack"])})
res = pd.DataFrame(rows); print(res)
res.plot(x="max_depth", y=["acc_train", "acc_test"], marker="o"); plt.show()

**Resultado esperado:** la exactitud de train sube hacia 1,0 con la profundidad; la de test se estanca en torno a 0,75–0,80. Esa brecha es el sobreajuste: el test incluye ataques que no están en train.

**Ojo:** no elijas la profundidad mirando el test. Se usa validación (sesión 7).

*Tu interpretación (mínimo 5 líneas):*

## Parte E · La trampa de la tasa base

In [ ]:
def alertas(n_eventos, tasa_base, tpr, fpr):
    ataques = n_eventos * tasa_base
    normales = n_eventos - ataques
    tp = ataques * tpr
    fp = normales * fpr
    return {"tp": tp, "fp": fp,
            "precision": tp / (tp + fp),
            "alertas": tp + fp}

print(alertas(1_000_000, 0.0001, 0.99, 0.01))
for fpr in [0.1, 0.01, 0.001, 0.0001]:
    print(fpr, alertas(1_000_000, 0.0001, 0.99, fpr)["precision"])

**Tarea:** ¿cuánto debe valer la FPR para que la precisión llegue al 50 % con esa tasa base?
*Tu respuesta:*

**Tres líneas:** qué significa para el equipo de un SOC y para elegir entre reglas y ML.
*Tu respuesta:*

## Parte F · Ficha de caso de uso (por parejas)
Elige un ámbito del mapa de usos y completa la ficha. Cada pareja la expone en 2 minutos.

| Campo | Vuestra respuesta |
|---|---|
| Ámbito elegido | |
| Datos: ¿qué fuente usaríais? | |
| Etiqueta: ¿qué es positivo y de dónde sale? | |
| Error caro: ¿falso negativo o falso positivo? | |
| Dataset público para empezar | |
| Evasión: ¿cómo engañaría un atacante al modelo? | |
| Técnicas ATT&CK que cubriría (attack.mitre.org) | |

## Parte G · Ampliación autónoma: aprende un umbral sin scikit-learn

In [ ]:
import numpy as np
cps = np.array([2, 3, 40, 55]); y4 = np.array([0, 0, 1, 1])
candidatos = np.r_[cps.min() - 1,
                   (cps[:-1] + cps[1:]) / 2,
                   cps.max() + 1]
for t in candidatos:
    pred = (cps > t).astype(int)
    print(f"t={t:6.1f} errores={int((pred != y4).sum())}")

**Preguntas:** ¿cuántos umbrales dan cero errores? ¿Cuál eliges? Añade un quinto flujo (`cps=25`, ataque): ¿qué cambia? Penaliza 5 veces más el falso negativo y recalcula el umbral.
*Tu respuesta:*

## Ampliación · Reglas contra modelo, y deriva
Correos sintéticos (semilla fija): una regla que solo mira la palabra *urgente* frente a una regresión logística con tres señales.

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score

rng = np.random.default_rng(7)
n = 5000
yb = rng.random(n) < 0.10                                # 10 % phishing
urgente = np.where(yb, rng.random(n) < 0.60, rng.random(n) < 0.05)
dom_dist = np.where(yb, rng.random(n) < 0.70, rng.random(n) < 0.03)
adj_exe = np.where(yb, rng.random(n) < 0.25, rng.random(n) < 0.01)
Xb = np.c_[urgente, dom_dist, adj_exe].astype(int)
Xtr, Xte, ytr, yte = train_test_split(Xb, yb, test_size=0.3, stratify=yb, random_state=7)

regla = Xte[:, 0] == 1
print("Regla  P=%.2f R=%.2f" % (precision_score(yte, regla), recall_score(yte, regla)))
modelo = LogisticRegression().fit(Xtr, ytr)
pred = modelo.predict(Xte)
print("Modelo P=%.2f R=%.2f" % (precision_score(yte, pred), recall_score(yte, pred)))
print(dict(zip(["urgente", "dominio_distinto", "adjunto_exe"], modelo.coef_[0].round(2))))

**Variante de deriva:** genera un segundo conjunto donde `dom_dist` en phishing baje de 0,70 a 0,20 (el atacante registra dominios coherentes) y mide de nuevo el recall del modelo ya entrenado. ¿Cuánto cae? ¿Qué acción tomarías?

## Autoevaluación (sin apuntes)
1. Diferencia entre IA, ML y DL, con un ejemplo de seguridad de cada una.
2. Por qué una exactitud del 99,9 % no demuestra que un detector sirva.
3. Tres cosas que pueden hacer fallar en tu red un modelo entrenado con NSL-KDD.
4. Qué significa que un modelo generalice.
5. Qué dato necesitas para conocer la precisión real de un detector en tu SOC.

## Antes de entregar
- **Colab:** *Entorno de ejecución › Reiniciar y ejecutar todo* y después *Archivo › Descargar › Descargar .ipynb*.
- **VS Code / Codespaces:** *Restart* y *Run All* en la barra del cuaderno; guarda y haz commit.
- Entrega `ia1_panorama.ipynb`, `utils.py` y la ficha de la parte F.